# Mini2P batch preprocessing, mean-image export and review PDFs

Run cells one at a time, not **Run All**. Don't point the QC notebook's `run_dir` inside the batch's derivatives folder: a run folder with a `.phase.json` but no manifest is treated as finished and skipped. Paths, session range and options come from the `[batch]` and `[exports]` sections of `config/pipeline.toml`.

The batch cell finds every raw TIFF matching `source_substring` in the session range, numbers them `run-01`, `run-02`, ... by acquisition counter, and runs the full pipeline (`mini2p-preprocess`) for each pending one in a fresh process. Each run folder gets a `source_tiff.json` manifest and a `preprocess.log`. Each variant has its own subfolder: `destripe = false` writes `run-XX/without_destriping/`, `destripe = true` writes `run-XX/with_destriping/`. To make both, run the batch once with each setting, in either order. Completed runs are skipped; failed or interrupted ones are retried if `retry_incomplete_runs`; conflicting assignments stop the launch instead of overwriting.

In [ ]:
import logging
from pathlib import Path

from mini2p_preprocessing.batch import format_plan, load_batch_settings, plan_batch, run_batch
from mini2p_preprocessing.config import variant_output_dir

logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s", force=True)
CONFIG = Path("../config/pipeline.toml").resolve()
batch = load_batch_settings(CONFIG)
plan = plan_batch(batch)
print(f"Destriping: {batch.destripe} -> run-XX/{variant_output_dir(Path(), batch.destripe).name}/")
print(format_plan(plan))

In [ ]:
LAUNCH = False  # check the plan above, then set True and run this cell
if LAUNCH:
    # Re-plans from the manifests on disk, so rerunning after a failure only
    # launches runs that are still pending, and freezes a copy of the config.
    run_batch(batch, CONFIG)
else:
    print("Dry run only.")

## Export fixed-contrast mean images for every run

Saves raw float TIFFs, contrast PNGs, a red/green overlay and a settings JSON beside each `ops.npy`. Requires exactly `expected_runs` Suite2p outputs. Existing files are kept unless `overwrite = true` (`pdf_overwrite` for the PDFs). Set `run_subdir` to the variant to export, `without_destriping` or `with_destriping`, and run these cells once per variant.

In [ ]:
from mini2p_preprocessing.exports import (
    export_all_mean_images,
    find_run_planes,
    load_export_settings,
    require_run_count,
    runs_with_both_settings_files,
    write_comparison_pdf,
    write_high_resolution_pdf,
)

exports = load_export_settings(CONFIG)
runs = find_run_planes(exports.subject_dir, exports.session_first, exports.session_last,
                       exports.run_subdir)
print(f"Found {len(runs)} Suite2p plane folders in run-XX/{exports.run_subdir}/.")
require_run_count(runs, exports.expected_runs)
summary = export_all_mean_images(runs, exports.subject_dir, exports.session_label,
                                 exports.contrast, exports.overwrite)
print("Summary:", summary)

## Review PDFs

One landscape page per run with green and red side by side, then a high-resolution PDF with green and red on separate portrait pages (contrast recomputed from `ops.npy`).

In [ ]:
both = runs_with_both_settings_files(runs)
print(f"Runs with both interactive and fixed contrast settings: {len(both)}")
for r in both:
    print(f"  ses-{r.session} {r.run_name}")

label = exports.session_label
comparison = write_comparison_pdf(
    runs, exports.subject_dir / f"meanImg_contrast_{label}.pdf", exports.contrast,
    f"Mini2P registered mean images, {label}", overwrite=exports.pdf_overwrite,
)
high_res = write_high_resolution_pdf(
    runs, exports.subject_dir / f"meanImg_high_resolution_{2 * len(runs)}_pages_{label}.pdf",
    exports.contrast, f"Mini2P high-resolution registered mean images, {label}",
    dpi=exports.high_resolution_dpi, overwrite=exports.pdf_overwrite,
)
print("Saved:", comparison)
print("Saved:", high_res)

## Extract the Cellpose masks into every run and review their alignment

For each run of the `run_subdir` variant, the masks in `run-XX/<cellpose_seg>` were drawn on `run-XX/<cellpose_image>`, a different registration of the same recording. They are aligned onto the variant's own mean image, then `stat`, `F`, `Fneu`, `iscell` and `spks` (and the red-channel traces) are written into its `suite2p/plane0`, as `mini2p-extract` does. Nothing is written for a run whose alignment is refused or whose masks come from a different TIFF (`failed`), or that has no masks or an unfinished batch run. A run already extracted from the same masks and settings is kept (`existing`); one extracted from other masks, e.g. in the QC notebook, is kept too (`other extraction`) unless `EXTRACT_OVERWRITE`. Masks re-saved in Cellpose under the same name are extracted again.

Writes `cellpose_extraction_summary_<label>.json` and `mask_alignment_<label>.pdf` (one page per run: the masks as segmented and as shifted, on the variant's mean image; for a refused run, the masks on the image they were drawn on and unshifted on this plane) in the subject folder. Run once per variant, like the exports.

In [ ]:
from mini2p_preprocessing.cellpose_batch import (
    ATTENTION,
    extract_all,
    format_summary,
    write_extraction_summary,
    write_mask_alignment_pdf,
)
from mini2p_preprocessing.config import load_settings

EXTRACT_OVERWRITE = False  # True also replaces extractions, including from other masks
rows = extract_all(exports.subject_dir, exports.session_first, exports.session_last,
                   exports.run_subdir, batch.cellpose_seg, batch.cellpose_image,
                   load_settings(CONFIG), exports.expected_runs, EXTRACT_OVERWRITE)
print(format_summary(rows))
attention = [r for r in rows if r["status"] in ATTENTION]
if attention:
    print(f"\n{len(attention)} run(s) NEED ATTENTION (no traces written from these masks):")
    for r in attention:
        print(f"  ses-{r['session']} {r['run']}: {r['status']}")
unverified = [f"ses-{r['session']} {r['run']}" for r in rows if r["source_check"] == "unverified"]
if unverified:  # run-XX/ has no record of its TIFF, so a run-number mix-up would not show
    print("Masks not checked against the run's TIFF:", ", ".join(unverified))

label = exports.session_label
print("Saved:", write_extraction_summary(
    rows, exports.subject_dir / f"cellpose_extraction_summary_{label}.json"))
print("Saved:", write_mask_alignment_pdf(  # a report rebuilt from the records, so replaced
    rows, exports.subject_dir / f"mask_alignment_{label}.pdf", exports.contrast,
    f"Cellpose masks on registered mean images, {label}", overwrite=True))
